# Notebook 07 — Gradio Demo

**Features:**
- Toggle RAG on/off
- Toggle between base and fine-tuned model
- Top-K slider
- Retrieved chunks display with source + score
- "Compare 4 configs" tab
- 6 example questions
- `demo.launch(share=True)` for 72-hour public Colab URL

> **Prerequisites:** Notebooks 01–04 must be complete.

In [8]:
import shutil

SOURCE_DRIVE = '/content/drive/MyDrive/NLP_Final/Source'
SOURCE_LOCAL = '/content/tdtu_project'

if not os.path.exists(SOURCE_LOCAL):
    print('📦 Copying entire project to local SSD...')
    shutil.copytree(SOURCE_DRIVE, SOURCE_LOCAL)
    print('✅ Done!')
else:
    print('✅ Project already in local SSD')

# Sau đó đổi BASE sang local
BASE = SOURCE_LOCAL
print(f'Base directory: {BASE}')

📦 Copying entire project to local SSD...
✅ Done!
Base directory: /content/tdtu_project


In [9]:
from pathlib import Path
import os

# if os.path.exists('/content/'):
#     from google.colab import drive
#     drive.mount('/content/drive')
#     BASE = '/content/drive/MyDrive/NLP_Final/Source'
# else:
#     BASE = '../'

BASE = '/content/tdtu_project'

print(f'Base directory: {BASE}')

Base directory: /content/tdtu_project


In [2]:
%%capture
!pip install -q trl==0.11.4 \
    transformers==4.45.0 peft==0.13.0 accelerate==1.0.1 \
    gradio==5.5.0
!pip install -q -U bitsandbytes
!pip install -q "numpy<2" faiss-cpu==1.8.0 sentence-transformers==3.2.1
print('Packages installed.')

## 1. Load All Components

In [10]:
import os, json, torch, pickle
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID  = 'Qwen/Qwen2.5-3B-Instruct'
SFT_PATH  = f'{BASE}/models/sft_checkpoint'
INDEX_DIR = f'{BASE}/data/vector_store/faiss_index'

SYSTEM_PROMPT = (
    'Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). '
    'Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. '
    'Trả lời bằng tiếng Việt. Nếu không có đủ thông tin, hãy nói rõ điều đó.'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_DTYPE = (
    torch.bfloat16 if device == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 30e9
    else torch.float16 if device == 'cuda'
    else torch.float32
)
print(f'Device: {device}  |  dtype: {COMPUTE_DTYPE}')

# --- FAISS ---
print('Loading FAISS index...')
faiss_index = faiss.read_index(f'{INDEX_DIR}/index.faiss')
with open(f'{INDEX_DIR}/index.pkl', 'rb') as f:
    index_chunks = pickle.load(f)

# --- Embedder ---
print('Loading embedding model...')
embedder = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')

# --- Tokenizer & Models ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(SFT_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

IM_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
base_model.eval()

print('Loading fine-tuned model...')
ft_model = PeftModel.from_pretrained(base_model, SFT_PATH)
ft_model.eval()

print('All components loaded.')

Device: cuda  |  dtype: torch.float16
Loading FAISS index...
Loading embedding model...
Loading tokenizer...
Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading fine-tuned model...
All components loaded.


## 2. Core Inference Functions

In [11]:
import numpy as np

def retrieve(query: str, k: int = 5) -> list[dict]:
    q_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = faiss_index.search(q_vec, k)
    return [{'chunk': index_chunks[i], 'score': float(scores[0][j])}
            for j, i in enumerate(indices[0]) if i < len(index_chunks)]


def build_prompt(question: str, context_chunks: list[str] | None) -> str:
    if context_chunks:
        context_str = '\n\n'.join(f'[{i+1}] {c}' for i, c in enumerate(context_chunks))
        user_content = (
            f'Dựa vào các đoạn văn bản sau từ quy chế TDTU:\n\n'
            f'{context_str}\n\nCâu hỏi: {question}'
        )
    else:
        user_content = question
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )


def generate(model, prompt: str, max_new_tokens: int = 300) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,          # Bật sampling
            temperature=0.7,         # Giảm temperature
            top_p=0.9,
            use_cache=True,          # Bật KV cache
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IM_END_ID,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print('Inference functions ready.')

Inference functions ready.


## 3. Gradio Interface

In [12]:
from trl import AutoModelForCausalLMWithValueHead
from transformers import AutoModelForSequenceClassification
from peft import PeftModel as PeftModelCls

PPO_PATH    = f'{BASE}/models/ppo_checkpoint'
REWARD_PATH = f'{BASE}/models/reward_model'

# --- PPO model ---
ppo_model = None
if os.path.exists(PPO_PATH):
    print('Loading PPO model (Config E)...')
    ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        PPO_PATH, torch_dtype=COMPUTE_DTYPE, device_map='auto',
    )
    ppo_model.pretrained_model.eval()
    print('PPO model loaded.')
else:
    print(f'PPO checkpoint not found at {PPO_PATH}')

# --- Reward model ---
reward_model     = None
reward_tokenizer = None
if os.path.exists(REWARD_PATH):
    print('Loading reward model...')
    reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_PATH, trust_remote_code=True)
    _reward_base = AutoModelForSequenceClassification.from_pretrained(
        'Qwen/Qwen2.5-3B-Instruct',
        num_labels=1,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    reward_model = PeftModelCls.from_pretrained(_reward_base, REWARD_PATH)
    reward_model.eval()
    print('Reward model loaded.')
else:
    print(f'Reward model not found at {REWARD_PATH}')

RLHF_AVAILABLE   = ppo_model is not None
REWARD_AVAILABLE = reward_model is not None


def reward_score(text: str) -> float:
    if not REWARD_AVAILABLE:
        return float('nan')
    inputs = reward_tokenizer(text, return_tensors='pt', truncation=True,
                              max_length=512).to(reward_model.device)
    with torch.no_grad():
        return reward_model(**inputs).logits.squeeze().item()

Loading PPO model (Config E)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PPO model loaded.
Loading reward model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-3B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward model loaded.


In [13]:
import gradio as gr

EXAMPLES_VI = [
    'Sinh viên bị đình chỉ học tập trong trường hợp nào?',
    'Điều kiện để được xét học bổng khuyến khích học tập là gì?',
    'Quy định về trang phục của sinh viên trong trường như thế nào?',
    'Sinh viên có thể chuyển ngành học không? Điều kiện và thủ tục ra sao?',
    'Mức xử lý kỷ luật khi sinh viên gian lận thi cử là gì?',
    'Điều kiện tốt nghiệp đại học tại TDTU là gì?',
]


def answer_question(question: str, use_rag: bool, use_finetuned: bool, top_k: int) -> tuple:
    if not question.strip():
        return 'Vui lòng nhập câu hỏi.', ''
    model = ft_model if use_finetuned else base_model
    if use_rag:
        results = retrieve(question, k=top_k)
        top3_texts = [r['chunk']['text'] for r in results[:3]]
        context_display = '\n\n'.join(
            f"**[{i+1}]** (Score: {r['score']:.3f}) *{r['chunk']['source_file']}*\n\n"
            f"{r['chunk']['text'][:300]}{'...' if len(r['chunk']['text']) > 300 else ''}"
            for i, r in enumerate(results)
        )
    else:
        top3_texts = []
        context_display = '*(RAG không được bật)*'
    prompt = build_prompt(question, top3_texts if use_rag else None)
    return generate(model, prompt), context_display


def compare_all_configs(question: str) -> tuple:
    if not question.strip():
        return ('Vui lòng nhập câu hỏi.',) * 4
    results = retrieve(question, k=5)
    top3_texts = [r['chunk']['text'] for r in results[:3]]
    ans_a = generate(base_model, build_prompt(question, None))
    ans_b = generate(base_model, build_prompt(question, top3_texts))
    ans_c = generate(ft_model,   build_prompt(question, None))
    ans_d = generate(ft_model,   build_prompt(question, top3_texts))
    return ans_a, ans_b, ans_c, ans_d


def compare_rlhf(question: str) -> tuple:
    if not question.strip():
        return 'Vui lòng nhập câu hỏi.', 'Vui lòng nhập câu hỏi.', ''
    results    = retrieve(question, k=5)
    top3_texts = [r['chunk']['text'] for r in results[:3]]
    prompt     = build_prompt(question, top3_texts)
    ans_d  = generate(ft_model, prompt)
    score_d = reward_score(ans_d)
    if RLHF_AVAILABLE:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=1024).to(ppo_model.pretrained_model.device)
        with torch.no_grad():
            outputs = ppo_model.pretrained_model.generate(
                **inputs, max_new_tokens=300, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=IM_END_ID,
            )
        ans_e = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                 skip_special_tokens=True).strip()
    else:
        ans_e = '(PPO model chưa có — chạy NB06 trước)'
    score_e = reward_score(ans_e)
    if REWARD_AVAILABLE and RLHF_AVAILABLE:
        score_info = (
            f'**Reward scores** (higher = better)\n\n'
            f'- Config D: `{score_d:.4f}`\n'
            f'- Config E (PPO): `{score_e:.4f}`\n'
            f'- Improvement: `{score_e - score_d:+.4f}`'
        )
    else:
        score_info = '*(Reward model hoặc PPO model chưa được load)*'
    return ans_d, ans_e, score_info


with gr.Blocks(title='TDTU QA System', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# Hệ thống Hỏi-Đáp Quy chế TDTU\n'
        '*Nhập môn Xử lý Ngôn ngữ Tự nhiên (504045) — Topic 1*'
    )

    with gr.Tab('Hỏi đáp'):
        with gr.Row():
            with gr.Column(scale=2):
                question_box = gr.Textbox(
                    label='Câu hỏi của bạn',
                    placeholder='Ví dụ: Điều kiện để được xét học bổng khuyến khích học tập là gì?',
                    lines=3,
                )
                with gr.Row():
                    use_rag_cb  = gr.Checkbox(label='Sử dụng RAG', value=True)
                    use_ft_cb   = gr.Checkbox(label='Sử dụng mô hình fine-tuned', value=True)
                    topk_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1, label='Top-K chunks')
                with gr.Row():
                    submit_btn = gr.Button('Gửi câu hỏi', variant='primary')
                    gr.ClearButton([question_box])
                gr.Examples(examples=EXAMPLES_VI, inputs=question_box)
            with gr.Column(scale=3):
                answer_box  = gr.Textbox(label='Câu trả lời', lines=8, interactive=False)
                context_box = gr.Markdown(label='Tài liệu tham khảo (RAG chunks)')
        submit_btn.click(
            fn=answer_question,
            inputs=[question_box, use_rag_cb, use_ft_cb, topk_slider],
            outputs=[answer_box, context_box],
        )

    with gr.Tab('So sánh 4 cấu hình (A/B/C/D)'):
        gr.Markdown(
            '**A:** Base, no RAG &nbsp;|&nbsp; **B:** Base + RAG &nbsp;|&nbsp; '
            '**C:** Fine-tuned, no RAG &nbsp;|&nbsp; **D:** Fine-tuned + RAG'
        )
        compare_q   = gr.Textbox(label='Nhập câu hỏi để so sánh', lines=2)
        compare_btn = gr.Button('So sánh tất cả', variant='secondary')
        with gr.Row():
            cfg_a_box = gr.Textbox(label='Config A (Base, no RAG)',        lines=7)
            cfg_b_box = gr.Textbox(label='Config B (Base + RAG)',           lines=7)
        with gr.Row():
            cfg_c_box = gr.Textbox(label='Config C (Fine-tuned, no RAG)', lines=7)
            cfg_d_box = gr.Textbox(label='Config D (Fine-tuned + RAG) ★', lines=7)
        compare_btn.click(
            fn=compare_all_configs,
            inputs=[compare_q],
            outputs=[cfg_a_box, cfg_b_box, cfg_c_box, cfg_d_box],
        )

    with gr.Tab('Config D vs PPO (Config E)'):
        rlhf_status = (
            '✅ PPO + Reward model sẵn sàng' if (RLHF_AVAILABLE and REWARD_AVAILABLE)
            else '⚠️ Cần chạy NB06 trước để có PPO và Reward model'
        )
        gr.Markdown(
            f'So sánh **Config D** (Fine-tuned + RAG) với **Config E** (PPO/RLHF + RAG)\n\n{rlhf_status}'
        )
        rlhf_q   = gr.Textbox(label='Câu hỏi', lines=2)
        rlhf_btn = gr.Button('So sánh D vs E', variant='secondary')
        with gr.Row():
            cfg_d_rlhf = gr.Textbox(label='D — Fine-tuned + RAG',   lines=8)
            cfg_e_rlhf = gr.Textbox(label='E — PPO (RLHF) + RAG ★', lines=8)
        score_md = gr.Markdown(label='Reward scores')
        rlhf_btn.click(
            fn=compare_rlhf,
            inputs=[rlhf_q],
            outputs=[cfg_d_rlhf, cfg_e_rlhf, score_md],
        )
        gr.Examples(examples=EXAMPLES_VI, inputs=rlhf_q)

    with gr.Tab('Thông tin mô hình'):
        ppo_status    = '✅ Available' if RLHF_AVAILABLE   else '⚠️ Not available'
        reward_status = '✅ Available' if REWARD_AVAILABLE else '⚠️ Not available'
        gr.Markdown(f'''
## Chi tiết mô hình

| Thành phần | Chi tiết |
|---|---|
| Base LLM | `Qwen/Qwen2.5-3B-Instruct` |
| Quantization | 4-bit NF4 (bitsandbytes) |
| Fine-tuning | QLoRA (r=16, alpha=32) via SFTTrainer |
| RLHF | PPO (2 epochs) |
| Reward model | LoRA r=8, `AutoModelForSequenceClassification` |
| Embedding | `bkai-foundation-models/vietnamese-bi-encoder` |
| Vector Store | FAISS IndexFlatIP |
| PPO model | {ppo_status} |
| Reward model | {reward_status} |
        ''')

print('Gradio interface built. Launching...')


Gradio interface built. Launching...


In [14]:
# share=True creates a 72-hour public URL accessible from anywhere
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4ad87472dc8cf13ad.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
APP_PY = '''
import os, json, torch, pickle
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"
HF_REPO     = "YOUR_HF_USERNAME/tdtu-qa-qwen2.5-3b-lora"

SYSTEM_PROMPT = (
    "Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). "
    "Trả lời bằng tiếng Việt, chính xác và đầy đủ."
)

# Load components at startup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
ft_model = PeftModel.from_pretrained(base_model, HF_REPO)
ft_model.eval()

faiss_index = faiss.read_index("faiss_index/index.faiss")
with open("faiss_index/index.pkl", "rb") as f:
    index_chunks = pickle.load(f)
embedder = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")


def retrieve(query, k=5):
    q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(q_vec, k)
    return [{"chunk": index_chunks[i], "score": float(scores[0][j])}
            for j, i in enumerate(indices[0]) if i < len(index_chunks)]


def answer(question, use_rag, use_finetuned, top_k):
    model = ft_model if use_finetuned else base_model
    if use_rag:
        results = retrieve(question, k=top_k)
        ctx = "\\n\\n".join(f"[{i+1}] {r[\'chunk\'][\'text\']}" for i, r in enumerate(results[:3]))
        user_content = f"Dựa vào các đoạn văn bản sau từ quy chế TDTU:\\n\\n{ctx}\\n\\nCâu hỏi: {question}"
        context_display = "\\n\\n".join(
            f"[{i+1}] Score={r[\'score\']:.3f} ({r[\'chunk\'][\' source_file\']}): {r[\'chunk\'][\'text\'][:200]}..."
            for i, r in enumerate(results)
        )
    else:
        user_content = question
        context_display = "(RAG không được bật)"
    prompt = f"<|im_start|>system\\n{SYSTEM_PROMPT}<|im_end|>\\n<|im_start|>user\\n{user_content}<|im_end|>\\n<|im_start|>assistant\\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return ans.split("<|im_end|>")[0].strip(), context_display


with gr.Blocks(title="TDTU QA") as demo:
    gr.Markdown("# Hệ thống Hỏi-Đáp Quy chế TDTU")
    q = gr.Textbox(label="Câu hỏi", lines=3)
    with gr.Row():
        rag_cb = gr.Checkbox(label="Sử dụng RAG", value=True)
        ft_cb  = gr.Checkbox(label="Fine-tuned model", value=True)
        k_sl   = gr.Slider(1, 10, value=5, step=1, label="Top-K")
    btn = gr.Button("Gửi", variant="primary")
    ans_box = gr.Textbox(label="Câu trả lời", lines=8)
    ctx_box = gr.Textbox(label="RAG chunks", lines=8)
    btn.click(fn=answer, inputs=[q, rag_cb, ft_cb, k_sl], outputs=[ans_box, ctx_box])

demo.launch()
'''

APP_PATH = f'{BASE}/app.py'
with open(APP_PATH, 'w', encoding='utf-8') as f:
    f.write(APP_PY)

print(f'app.py saved → {APP_PATH}')
print('To deploy on HuggingFace Spaces:')
print('  1. Create a new Space (SDK: Gradio)')
print('  2. Upload app.py, faiss_index/, and requirements.txt')
print('  3. Set MODEL_ID and HF_REPO in app.py')

app.py saved → /content/drive/MyDrive/NLP_Final/Source/app.py
To deploy on HuggingFace Spaces:
  1. Create a new Space (SDK: Gradio)
  2. Upload app.py, faiss_index/, and requirements.txt
  3. Set MODEL_ID and HF_REPO in app.py
